# IdiomX – Task 2 Inference Demo

This notebook provides a lightweight inference pipeline for **Task 2: Context → Idiom Retrieval**.

It loads the saved retrieval artifacts from the main benchmark notebook and applies the final **Hybrid + Reranker** pipeline to new input sentences.

## What this notebook does

- loads the idiom bank and saved embeddings
- rebuilds the lexical index
- loads the dense embedder and reranker
- predicts the most likely idioms for one or more English sentences

This notebook is designed for simple demonstration and quick testing.

In [15]:
# [B1.1] Core imports

from pathlib import Path
import warnings
import pickle
import sys
import subprocess

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("Core imports loaded.")

Core imports loaded.


In [16]:
# [B1.2] Ensure required libraries are available

# sentence-transformers
try:
    from sentence_transformers import SentenceTransformer, CrossEncoder
    print("sentence-transformers already installed.")
except ImportError:
    print("Installing sentence-transformers...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
    from sentence_transformers import SentenceTransformer, CrossEncoder
    print("sentence-transformers installed and imported.")

# rank_bm25
try:
    from rank_bm25 import BM25Okapi
    print("rank_bm25 already installed.")
except ImportError:
    print("Installing rank_bm25...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rank_bm25"])
    from rank_bm25 import BM25Okapi
    print("rank_bm25 installed and imported.")

sentence-transformers already installed.
rank_bm25 already installed.


### Loading Saved Artifacts

We load the saved Task 2 retrieval artifacts and validate that all required files are available before running inference.

In [17]:
# [B1.3] Load and validate saved artifacts from Notebook A

ARTIFACT_DIR = Path("../artifacts/task2")

# Check that all required files exist before loading
required_files = [
    "idiom_bank.csv",
    "idiom_embeddings.npy",
    "idiom_to_idx.pkl",
    "idx_to_idiom.pkl"
]

missing_files = [file_name for file_name in required_files if not (ARTIFACT_DIR / file_name).exists()]

if missing_files:
    raise FileNotFoundError(
        f"Missing artifacts: {missing_files}\n"
        f"Please run the benchmark notebook first to generate them."
    )

# Load idiom bank
idiom_bank = pd.read_csv(ARTIFACT_DIR / "idiom_bank.csv")

# Load saved idiom embeddings
idiom_embeddings = np.load(ARTIFACT_DIR / "idiom_embeddings.npy")

# Load idiom index mappings
with open(ARTIFACT_DIR / "idiom_to_idx.pkl", "rb") as f:
    idiom_to_idx = pickle.load(f)

with open(ARTIFACT_DIR / "idx_to_idiom.pkl", "rb") as f:
    idx_to_idiom = pickle.load(f)

print("Artifacts loaded successfully.")
print("Idiom bank size:", len(idiom_bank))
print("Embedding shape:", idiom_embeddings.shape)

Artifacts loaded successfully.
Idiom bank size: 11626
Embedding shape: (11626, 384)


In [18]:
# [B1.4] Build BM25 index from idiom bank

# We use the canonical idiom text (same as training)
idioms_list = idiom_bank["idiom_canonical"].astype(str).tolist()

# Tokenize (simple whitespace, same logic as main notebook)
tokenized_idioms = [idiom.lower().split() for idiom in idioms_list]

# Build BM25 index
bm25 = BM25Okapi(tokenized_idioms)

print("BM25 index built.")
print("Number of indexed idioms:", len(tokenized_idioms))

BM25 index built.
Number of indexed idioms: 11626


In [19]:
# [B1.5] Load the final embedding model and reranker

# Dense embedder used for semantic retrieval
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Cross-encoder used for reranking top candidates
reranker_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Embedder and reranker loaded.")

Embedder and reranker loaded.


In [20]:
# [B1.6] Define the Hybrid + Reranker inference pipeline with clean table output

def minmax_normalize(scores):
    """Normalize scores to the [0, 1] range."""
    scores = np.asarray(scores, dtype=float)
    return (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)


def predict_idiom(query_text, top_k=3, rerank_top_k=10):
    """
    Run Hybrid + Reranker inference for one query.
    
    Returns a clean pandas DataFrame with:
    - final rank
    - predicted idiom
    - normalized final score in [0, 1]
    """
    
    # 1. Encode the input query
    query_embedding = embedder.encode(
        [query_text],
        convert_to_numpy=True
    )[0]

    # 2. Compute dense similarity scores
    dense_scores = np.dot(query_embedding, idiom_embeddings.T)

    # 3. Compute lexical BM25 scores
    bm25_scores = np.array(
        bm25.get_scores(query_text.lower().split()),
        dtype=float
    )

    # 4. Normalize both score types
    dense_scores_norm = minmax_normalize(dense_scores)
    bm25_scores_norm = minmax_normalize(bm25_scores)

    # 5. Combine dense + lexical scores
    hybrid_scores = 0.5 * dense_scores_norm + 0.5 * bm25_scores_norm

    # 6. Keep only top candidates for reranking
    top_candidate_indices = np.argsort(-hybrid_scores)[:rerank_top_k]
    top_candidate_idioms = [idx_to_idiom[idx] for idx in top_candidate_indices]

    # 7. Build query–candidate pairs
    rerank_pairs = [[query_text, idiom] for idiom in top_candidate_idioms]

    # 8. Score pairs with the reranker
    rerank_scores = reranker_model.predict(
        rerank_pairs,
        batch_size=32,
        show_progress_bar=False
    )

    # 9. Normalize reranker scores for final display
    final_scores = minmax_normalize(rerank_scores)

    # 10. Build result table
    result_df = pd.DataFrame({
        "idiom_prediction": top_candidate_idioms,
        "score": final_scores
    })

    # 11. Sort by final normalized score
    result_df = result_df.sort_values(
        by="score",
        ascending=False
    ).reset_index(drop=True)

    # 12. Add ranking column
    result_df.insert(0, "rank", range(1, len(result_df) + 1))

    # 13. Return only top-k rows
    return result_df.head(top_k)

---

In [21]:
# [B1.7] Demo: single query with table output

query_text = "He finally gave up after trying the same thing again and again."

result_df = predict_idiom(query_text, top_k=3)

print(f"Query: {query_text}")
result_df

Query: He finally gave up after trying the same thing again and again.


,rank,idiom_prediction,score
0,1,after one thing,1.000000
1,2,once again,0.823268
2,3,then again,0.614177


---

### Multi-Query Demo

We can also test several input sentences in sequence to inspect the model’s behavior across different contexts.

In [22]:
# [B1.8] Demo: multiple queries

demo_queries = [
    "She finally revealed the secret to everyone.",
    "After so many setbacks, he decided to quit.",
    "He avoided answering the question directly.",
]

for query_text in demo_queries:
    print(f"\nQuery: {query_text}")
    display(predict_idiom(query_text, top_k=3))


Query: She finally revealed the secret to everyone.


,rank,idiom_prediction,score
0,1,and finally,1.000000
1,2,top secret,0.759345
2,3,best-kept secret,0.657344



Query: After so many setbacks, he decided to quit.


,rank,idiom_prediction,score
0,1,after a while,1.000000
1,2,ever after,0.819766
2,3,after one thing,0.816178



Query: He avoided answering the question directly.


,rank,idiom_prediction,score
0,1,out of the question,1.000000
1,2,no question,0.906015
2,3,that's what he said,0.767245


### Demo Note

These examples are intentionally more open-ended than the earlier test cases.

They show that the pipeline is strongest when the context clearly signals a known idiom, while more generic sentences may retrieve semantically related expressions instead of a single exact target.

In [23]:
# [B1.9] Run inference on a custom list of sentences

demo_queries = [
    "He refused to reveal the details and kept everything secret.",
    "After so many failures, she finally gave up.",
    "The negotiations were difficult, but both sides reached an agreement.",
    "He was extremely nervous before the interview.",
    "She said something that made the whole room suddenly silent."
]

for query_text in demo_queries:
    print(f"\nQuery: {query_text}")
    display(predict_idiom(query_text, top_k=3))


Query: He refused to reveal the details and kept everything secret.


,rank,idiom_prediction,score
0,1,spare someone the details,1.000000
1,2,everything is everything,0.849707
2,3,that's what he said,0.842864



Query: After so many failures, she finally gave up.


,rank,idiom_prediction,score
0,1,and finally,1.000000
1,2,after all,0.997857
2,3,ever after,0.749826



Query: The negotiations were difficult, but both sides reached an agreement.


,rank,idiom_prediction,score
0,1,my sides,1.000000
1,2,split one's sides,0.981061
2,3,take sides,0.901858



Query: He was extremely nervous before the interview.


,rank,idiom_prediction,score
0,1,before it was cool,1.000000
1,2,nervous hit,0.990863
2,3,not before time,0.985057



Query: She said something that made the whole room suddenly silent.


,rank,idiom_prediction,score
0,1,there is something to be said for,1.000000
1,2,read the room,0.961778
2,3,There she blows,0.841048


### Conclusion

This notebook demonstrates a lightweight inference pipeline for Task 2: Context → Idiom Retrieval.

The system combines:
- dense semantic retrieval (MiniLM)
- lexical matching (BM25)
- cross-encoder reranking

The final predictions are presented as ranked candidates with normalized confidence scores.

While the model performs strongly when the input clearly matches a known idiom, more general or ambiguous sentences may return semantically related expressions rather than a single exact idiom.

This behavior reflects the inherent difficulty of open-ended idiom retrieval tasks.

In [24]:
from pathlib import Path
import pandas as pd
import pickle
from rank_bm25 import BM25Okapi

ARTIFACT_DIR = Path("../artifacts/task2")

idiom_bank = pd.read_csv(ARTIFACT_DIR / "idiom_bank.csv")

TEXT_COL = "idiom_canonical"   # change only if needed

tokenized_corpus = [
    str(text).lower().split()
    for text in idiom_bank[TEXT_COL].fillna("")
]

bm25 = BM25Okapi(tokenized_corpus)

with open(ARTIFACT_DIR / "bm25.pkl", "wb") as f:
    pickle.dump(bm25, f)

print("BM25 rebuilt and saved to:", ARTIFACT_DIR / "bm25.pkl")

BM25 rebuilt and saved to: ..\artifacts\task2\bm25.pkl
